In [5]:
import faiss
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from ast import literal_eval
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
from scipy.stats import zscore, entropy
import json

def print_ix(df, ix, c1='query_sent', c2='target_sent'):
    print(df[c1][ix])
    print(df[c2][ix])

In [27]:
# --- Load metadata first ---
with open('data/meta-kst.json', 'r') as f:
    kst_proc_ids = json.load(f)

with open('data/meta-reports.json', 'r') as f:
    report_ids = json.load(f)

In [29]:
Counter([m['sub_source'] for m in kst_proc_ids])

Counter({'Bijlage': 110290,
         'Brief regering': 84682,
         'Motie': 54551,
         'Schriftelijke vragen': 44917,
         'Antwoord schriftelijke vragen': 43372,
         'Verslag van een algemeen overleg': 5131,
         'Verslag van een schriftelijk overleg': 4525,
         'Brief commissie': 3354,
         'Verslag van een commissiedebat': 1181,
         'Verslag van een wetgevingsoverleg': 681,
         'Antwoord schriftelijke vragen (nader)': 172})

In [7]:
# --- Identify relevant IDs early ---
kst_proc_ids = {
    m['id_'] for m in kst_proc_ids
    if 'verslag' in m['sub_source'].lower()
}

report_ids = {
    m['_id'] for m in report_ids
    if not any(e in m['_id'].lower() for e in {"verantw", "jaarverslag", ":vo-", "trendrapport"})
}

# --- Load metadata-filtered dataframe ---
usecols = ['published_at', 'sid', 'sentences', '_id']
df = pd.read_csv('data/subset/sentences.tsv', sep='\t', usecols=usecols)

# Keep only rows relevant to filtered metadata
mask = (
    df.sid.str.split('_s').str[0].isin(kst_proc_ids)
    | df['_id'].isin(report_ids)
)
df = df[mask].reset_index(drop=False)  # keep original FAISS indices
df = df[df.sentences.str.split(' ').str.len() >= 15]

In [8]:
# --- Load FAISS index ---
index = faiss.read_index('data/sentences.faiss')

# --- Reconstruct only needed embeddings ---
# Note: df['index'] is the original row index (FAISS vector position)
xb = np.vstack([index.reconstruct(int(i)) for i in df['index']]).astype('float32')
xb = normalize(xb, axis=1)

del index

In [9]:
report_vecs = xb[df['_id'].isin(report_ids)]
kst_vecs = xb[df.sid.str.split('_s').str[0].isin(kst_proc_ids)]

report_df = df[df['_id'].isin(report_ids)].reset_index(drop=True)
kst_df = df[df.sid.str.split('_s').str[0].isin(kst_proc_ids)].reset_index(drop=True)

In [10]:
def compute_averageness(vectors):
    """Compute cosine similarity to corpus mean (semantic centrality)."""
    mean_vec = np.mean(vectors, axis=0)
    mean_vec /= np.linalg.norm(mean_vec)
    norms = np.linalg.norm(vectors, axis=1)
    sims = (vectors @ mean_vec) / norms
    return sims

report_averageness = compute_averageness(report_vecs)
kst_averageness = compute_averageness(kst_vecs)

# Attach to your DataFrames (optional)
report_df["averageness"] = report_averageness
kst_df["averageness"] = kst_averageness

In [11]:
report_vecs_av = pd.DataFrame(report_vecs).groupby(report_df._id).mean()
kst_vecs_av = pd.DataFrame(kst_vecs).groupby(kst_df._id).mean()

# Compute all pairwise similarities
av_similarity_matrix = cosine_similarity(report_vecs_av, kst_vecs_av)

# Convert to a DataFrame for easier viewing
av_similarities_df = pd.DataFrame(
    av_similarity_matrix,
    index=report_vecs_av.index,  # report IDs
    columns=kst_vecs_av.index    # KST IDs
)

# If you want to "melt" this into a long format (report_id, kst_id, similarity)
all_similarities = av_similarities_df.reset_index().melt(
    id_vars='_id', var_name='kst_id', value_name='similarity'
)

In [12]:
all_similarities = {(r['_id'],r['kst_id']):r['similarity'] for i,r in all_similarities.iterrows()}

In [13]:
report_dates = pd.to_datetime(report_df['published_at']).to_numpy(dtype='datetime64[ns]')
kst_dates = pd.to_datetime(kst_df['published_at']).to_numpy(dtype='datetime64[ns]')

In [14]:
threshold = 0.25
batch_size = 400  # Adjust depending on your memory
results = []

for start in tqdm(range(0, len(report_vecs), batch_size)):
    end = min(start + batch_size, len(report_vecs))
    batch_vecs = report_vecs[start:end]
    batch_dates = report_dates[start:end]

    # Compute similarity for the batch: (batch_size x num_targets)
    sims = batch_vecs @ kst_vecs.T

    # Date mask: only consider targets after query
    date_mask = batch_dates[:, None] < kst_dates[None, :]
    sims_masked = np.where(date_mask, sims, -1.0)

    # Find indices above threshold
    query_idx, target_idx = np.where(sims_masked > threshold)

    if len(query_idx) > 0:
        similarity_batch = pd.DataFrame({
            'query_id': report_df['_id'].values[start:end][query_idx],
            'query_sent': report_df['sentences'].values[start:end][query_idx],
            'query_date': report_df['published_at'].values[start:end][query_idx],
            'query_averageness': report_averageness[start:end][query_idx],

            'target_id': kst_df['_id'].values[target_idx],
            'target_sent': kst_df['sentences'].values[target_idx],
            'target_date': kst_df['published_at'].values[target_idx],
            'target_averageness': kst_averageness[target_idx],

            'similarity': sims_masked[query_idx, target_idx]
        })
        results.append(similarity_batch)

# Combine all batches
similarity_df = pd.concat(results, ignore_index=True)

100%|██████████| 190/190 [05:28<00:00,  1.73s/it]


In [25]:
dft = similarity_df[(similarity_df.similarity>.75)]
dft.shape

(37, 9)

In [26]:
dft

,query_id,query_sent,query_date,query_averageness,target_id,target_sent,target_date,target_averageness,similarity
5077548,rekenkamer__rapport:2003:02:07:vbtb-toets-rege...,Dat wil zeggen dat wordt neergelegd wat het ka...,2003-02-07,0.062990,kamerstukken__wetgevingsoverleg:7080b6db-67b6-...,Afgesproken is dat daarbij duidelijk wordt aan...,2008-12-17,0.192429,0.798541
5670814,rekenkamer__rapport:2003:11:06:opvang-zwerfjon...,Tenslotte zijn gesprekken gevoerd met het Mini...,2003-11-06,0.100162,kamerstukken__algemeen_overleg:065ce9cd-a1a3-4...,"De minister gaat praten met de ministers, met ...",2010-02-10,0.063275,0.821185
5810776,rekenkamer__rapport:2004:02:04:beveiliging-mil...,"Echter, om te voorkomen dat de aandacht voor b...",2004-02-04,0.328195,kamerstukken__algemeen_overleg:bd24949e-6835-4...,Ik citeer: «Om te voorkomen dat de aandacht vo...,2008-12-15,0.427187,0.765534
6896207,rekenkamer__rapport:2005:05:18:staat-van-de-be...,6 VBTB vereist dat de begroting antwoord moete...,2005-05-18,0.025051,kamerstukken__algemeen_overleg:6218fd8e-acd2-4...,Vooraf beantwoord je de drie W-vragen: wat wil...,2018-05-31,0.158731,0.761892
6897024,rekenkamer__rapport:2005:05:18:staat-van-de-be...,Het jaarverslag moet antwoord geven op de drie...,2005-05-18,0.089629,kamerstukken__algemeen_overleg:6218fd8e-acd2-4...,Achteraf leg je verantwoording af door beantwo...,2018-05-31,0.201367,0.814767
8869312,rekenkamer__rapport:2007:11:29:lessen-uit-ict-...,Bij deze te complexe projecten is er geen bala...,2007-11-29,0.036816,kamerstukken__algemeen_overleg:a2cb95e8-cbae-4...,Volgens de Algemene Rekenkamer is er bij deze ...,2008-09-30,0.009043,0.910337
8877515,rekenkamer__rapport:2007:11:29:lessen-uit-ict-...,Wij vatten deze aanbevelingen samen als: wees ...,2007-11-29,-0.025126,kamerstukken__algemeen_overleg:a2cb95e8-cbae-4...,De Algemene Reken- kamer vat het als volgt sam...,2008-09-30,0.072298,0.772890
9704046,rekenkamer__rapport:2008:07:01:lessen-uit-ict-...,Bij deze te complexe projecten is er geen bala...,2008-07-01,0.036816,kamerstukken__algemeen_overleg:a2cb95e8-cbae-4...,Volgens de Algemene Rekenkamer is er bij deze ...,2008-09-30,0.009043,0.910337
9737705,rekenkamer__rapport:2008:07:01:lessen-uit-ict-...,Zij geeft aan wel systeemverantwoordelijk te z...,2008-07-01,0.123810,kamerstukken__algemeen_overleg:1b32d419-14f5-4...,De minister geeft aan dat zij systeemverantwoo...,2008-11-03,-0.006526,0.801931
9738029,rekenkamer__rapport:2008:07:01:lessen-uit-ict-...,Zij ziet echter wel mogelijkheden om de systee...,2008-07-01,0.122555,kamerstukken__algemeen_overleg:1b32d419-14f5-4...,De minister acht het wel opportuun om de syste...,2008-11-03,0.121192,0.753789


In [16]:
# dft = similarity_df[(similarity_df.similarity>.3)]
dft['month_diff'] = (pd.to_datetime(dft['target_date']).dt.year - pd.to_datetime(dft['query_date']).dt.year) * 12 + \
                   (pd.to_datetime(dft['target_date']).dt.month - pd.to_datetime(dft['query_date']).dt.month)
dft = dft[(dft.month_diff<180) &
          ~(dft.query_sent.str.lower().str.contains('minister')) &
          ~(dft.query_sent.str.lower().str.contains('voorzitter')) &
          ~(dft.query_sent.str.lower().str.contains('figuur')) &
          ~(dft.query_sent.str.lower().str.contains('vergaderjaar')) &
          ~(dft.query_sent.str.lower().str.contains('titel')) &
          ~(dft.target_sent.str.lower().str.contains('titel'))]

dft['kst_type'] = dft.target_id.str.split('__').str[1].str.split(':').str[0]
dft['mdr'] = dft.month_diff // 12 * 12
print(len(dft))

1234


/tmp/ipykernel_1177702/901386199.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dft['month_diff'] = (pd.to_datetime(dft['target_date']).dt.year - pd.to_datetime(dft['query_date']).dt.year) * 12 + \


In [17]:
dft.to_csv('results/matches-e5-finetuned.csv',sep='\t',index=False)